# Phase 5 — Advanced techniques + ablation + frontier-LLM head-to-head
**AI Agent Conversation Quality Scorer · HaluEval-QA · primary metric = length-matched macro-F1**

### Where Phase 4 left us
The champion `eng_xgboost` (13 features: `ground_overlap` ⊕ 12 engineered) sits at **0.9808 matched macro-F1 / 0.9967 raw** and is *feature-bound and saturated*: Optuna moved it +0.0000, XGB=LGBM=CatBoost returned the identical number, calibration was a no-op (Platt made it worse), and 20× more data saturates at 5%. The **only** residual failure mode is a different problem entirely — **question-relevance, not knowledge-grounding**: 10 of the champion's 12 false negatives are *verbatim substrings of the knowledge* scored P≈0.01 — grounded to the passage, but the **wrong answer to the question** (e.g. Q about *"From Eden"* → hallucinated answer *"Take Me To Church"*, another Hozier song quoted in the passage).

### Phase 5 asks three questions
1. **Can a question-aware signal break the ceiling?** Engineer a model that reads the *question*, not just the passage — a cross-encoder fine-tuned with the question in its input (QA-aware), vs the Phase-3 grounding-only CE. Does it rescue the *grounded-but-irrelevant* residual that no grounding feature can?
2. **Ablation — which features actually carry the signal, and does any feature HURT?** Leave-one-feature-out over all 13.
3. **Frontier head-to-head.** Claude Opus 4.8, Claude Haiku 4.5, Codex GPT-5.5 — zero-shot vs the champion. Headline metrics on a representative n=50; then the **grounded-but-irrelevant probe** as the real test: do frontier LLMs, which *reason* about question relevance, catch what a 13-feature tree cannot?

### Research that shaped this phase (5a)
- **Luna** (Belyi et al., *arXiv 2406.00975*, 2024) — a small fine-tuned DeBERTa-NLI + shallow classifier, conditioned on **query + context**, catches LLM hallucinations at high accuracy and ~100–1000× lower cost than GPT judges. Motivates both the QA-aware CE and the cost framing of the LLM head-to-head.
- **QA-based faithfulness** (QAFactEval / QuestEval lineage) — faithfulness is verified by *asking questions of the answer and checking against the source*; relevance is undecidable without the question. Direct motivation for putting the question inside the encoder.
- **"The Illusion of Progress: Re-evaluating Hallucination Detection in LLMs"** (*arXiv 2508.08285*, 2025) — detectors drop up to 45.9% under human-aligned metrics; *how* you evaluate decides who wins. Keeps the head-to-head honest: latency includes CLI overhead, F1 is real predictions, and we probe the specific failure mode rather than only a headline average.

In [1]:
import json, os, re, time, warnings, difflib, bisect
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
warnings.filterwarnings("ignore")
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                             precision_score, recall_score, roc_auc_score, confusion_matrix)
from scipy.stats import ks_2samp
import sklearn, scipy, torch
from xgboost import XGBClassifier
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("sklearn", sklearn.__version__, "| torch", torch.__version__, "| device", DEVICE)

def find_root():
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / "data" / "raw" / "qa_data.json").exists():
            return cand
    raise RuntimeError("repo root not found")
ROOT = find_root(); RESULTS = ROOT/"results"; CACHE = RESULTS/"phase5_cache"; MODELS = ROOT/"models"
CACHE.mkdir(parents=True, exist_ok=True); (CACHE/"llm").mkdir(exist_ok=True)
print("root:", ROOT)

sklearn 1.8.0 | torch 2.11.0 | device mps
root: /Users/anthonyrodrigues/Desktop/YC-Portfolio-Projects/AI-Agent-Conversation-Quality-Scorer


## 1 · Data, frozen split, and the length-matched control
Identical pipeline to Phases 1–4: each HaluEval-QA item → grounded(0) + hallucinated(1); the same frozen `GroupShuffleSplit` (group=qid) from Phase 1; and the per-answer-length nearest-neighbour control (caliper=8 chars) that strips HaluEval's length shortcut. Every leaderboard is ranked by **length-matched** macro-F1.

In [2]:
raw_path = ROOT/"data"/"raw"/"qa_data.json"
records = [json.loads(l) for l in raw_path.read_text().splitlines() if l.strip()]
long = []
for qid, r in enumerate(records):
    long.append({"qid": qid, "knowledge": r["knowledge"], "question": r["question"], "answer": r["right_answer"], "label": 0})
    long.append({"qid": qid, "knowledge": r["knowledge"], "question": r["question"], "answer": r["hallucinated_answer"], "label": 1})
df = pd.DataFrame(long).reset_index(drop=True)
df["ans_chars"] = df.answer.str.len()

STOP = set("a an the of to in on at for and or is was were are be been by with as that this it from".split())
def toks(s):     return [t for t in re.findall(r"[a-z0-9]+", str(s).lower()) if t not in STOP]
def toks_all(s): return re.findall(r"[a-z0-9]+", str(s).lower())
def grounding_overlap(ans, know):
    a = set(toks(ans)); k = set(toks(know)); return (len(a & k)/len(a)) if a else 0.0
df["ground_overlap"] = [grounding_overlap(a,k) for a,k in zip(df.answer, df.knowledge)]

split = json.load(open(RESULTS/"phase1_split_qids.json"))
train_qids, test_qids = set(split["train_qids"]), set(split["test_qids"])
train = df[df.qid.isin(train_qids)]; test = df[df.qid.isin(test_qids)]

def nearest_length_match(frame, caliper=8):
    g0 = frame[frame.label==0].sort_values("ans_chars"); g1 = frame[frame.label==1].sort_values("ans_chars")
    chars0 = g0.ans_chars.tolist(); idx0 = g0.index.tolist(); keep0, keep1 = [], []
    for c1, i1 in zip(g1.ans_chars.tolist(), g1.index.tolist()):
        if not chars0: break
        p = bisect.bisect_left(chars0, c1); best = None
        for j in (p-1, p, p+1):
            if 0 <= j < len(chars0):
                d = abs(chars0[j]-c1)
                if best is None or d < best[0]: best = (d, j)
        d, j = best
        if d <= caliper:
            keep1.append(i1); keep0.append(idx0[j]); del chars0[j]; del idx0[j]
    return frame.loc[keep0 + keep1]
matched = nearest_length_match(test, caliper=8)
def ks(fr): return ks_2samp(fr[fr.label==0].ans_chars, fr[fr.label==1].ans_chars).statistic
print(f"train {len(train)} | test {len(test)} | matched control n={len(matched)}")
print(f"answer-length KS  raw={ks(test):.3f} -> matched={ks(matched):.3f}  (shortcut removed)")

def evaluate(name, fr, y_pred, y_score, splitname):
    y = fr.label
    return {"model": name, "split": splitname, "n": int(len(fr)),
            "accuracy": round(accuracy_score(y, y_pred), 4),
            "macro_f1": round(f1_score(y, y_pred, average="macro"), 4),
            "balanced_acc": round(balanced_accuracy_score(y, y_pred), 4),
            "precision_hallu": round(precision_score(y, y_pred, pos_label=1, zero_division=0), 4),
            "recall_hallu": round(recall_score(y, y_pred, pos_label=1, zero_division=0), 4),
            "roc_auc": round(roc_auc_score(y, y_score), 4) if y_score is not None and len(set(y))>1 else None}
def macro(fr, pred): return f1_score(fr.label, pred, average="macro")
BAR_P4 = 0.9808
print("Phase-4 champion bar to beat (matched macro-F1):", BAR_P4)

train 16000 | test 4000 | matched control n=572
answer-length KS  raw=0.874 -> matched=0.122  (shortcut removed)
Phase-4 champion bar to beat (matched macro-F1): 0.9808


## 2 · Reproduce the Phase-4 champion and isolate the residual
Rebuild the 12 engineered features (`feat_row`, identical to Phase 3) + `ground_overlap` = the 13-feature stack, refit `eng_xgboost` with the frozen hyper-parameters, and confirm we recover **0.9808 matched / 0.9967 raw**. Then isolate the **grounded-but-irrelevant residual**: hallucinations the champion scores as grounded *because the answer is verbatim text from the passage* — the wrong answer to the question. This residual is the probe target for the rest of the phase.

In [3]:
NUM_RE  = re.compile(r"\d+(?:\.\d+)?")
YEAR_RE = re.compile(r"\b(?:1[0-9]{3}|20[0-9]{2})\b")
NEG = {"not","no","never","none","cannot","without","neither","nor","n't","dont","didnt","doesnt","isnt","wasnt","werent","arent","wont","cant"}
def split_sents(txt):
    s = re.split(r"(?<=[.!?])\s+", str(txt).strip()); return [x for x in s if x.strip()] or [str(txt)]
docfreq = Counter()
for s in pd.concat([train.knowledge, train.answer]):
    docfreq.update(set(toks(s)))
NDOC = 2*len(train)
def idf(t): return np.log((NDOC+1)/(docfreq.get(t,0)+1))

def feat_row(ans, q, know):
    a_lc = str(ans).lower().strip().rstrip("."); k_lc = str(know).lower()
    aset = set(toks(ans)); kset = set(toks(know))
    at_all = toks_all(ans); kt_all = toks_all(know)
    is_substr = float(a_lc in k_lc) if a_lc else 0.0
    lcs_char = (difflib.SequenceMatcher(None, a_lc, k_lc, autojunk=False)
                .find_longest_match(0, len(a_lc), 0, len(k_lc)).size / len(a_lc)) if a_lc else 0.0
    lcs_tok = (difflib.SequenceMatcher(None, at_all, kt_all, autojunk=False)
               .find_longest_match(0, len(at_all), 0, len(kt_all)).size / len(at_all)) if at_all else 0.0
    anum = set(NUM_RE.findall(str(ans))); knum = set(NUM_RE.findall(str(know)))
    num_frac = (len(anum & knum)/len(anum)) if anum else 1.0
    n_miss = len(anum - knum)
    ayr = set(YEAR_RE.findall(str(ans))); year_mismatch = len(ayr - set(YEAR_RE.findall(str(know))))
    sent_max = 0.0
    if aset:
        for s in split_sents(know):
            ss = set(toks(s));  o = len(aset & ss)/len(aset) if ss else 0.0
            if o > sent_max: sent_max = o
    whole = (len(aset & kset)/len(aset)) if aset else 0.0
    spread = whole - sent_max
    novel = aset - set(toks(q))
    novel_ov = (len(novel & kset)/len(novel)) if novel else 1.0
    num = sum(idf(t) for t in aset if t in kset); den = sum(idf(t) for t in aset)
    idf_ov = (num/den) if den else 0.0
    ans_neg = float(any(w in NEG for w in at_all) or "n't" in str(ans).lower())
    best_s, best = "", -1.0
    if aset:
        for s in split_sents(know):
            ss = set(toks(s)); o = len(aset & ss)/len(aset) if ss else 0.0
            if o > best: best, best_s = o, s
    sent_neg = float(any(w in NEG for w in toks_all(best_s)) or "n't" in best_s.lower())
    neg_mismatch = float(ans_neg != sent_neg)
    return (is_substr, lcs_char, lcs_tok, num_frac, n_miss, year_mismatch,
            sent_max, spread, novel_ov, idf_ov, ans_neg, neg_mismatch)

ENG = ["is_substr","lcs_char_ratio","lcs_token_ratio","num_frac_in_know","n_num_missing","year_mismatch",
       "sent_overlap_max","overlap_spread","novel_overlap","idf_overlap","ans_has_neg","neg_mismatch"]
fp = CACHE/"eng_F.npy"
t0 = time.time()
F = np.array([feat_row(a,q,k) for a,q,k in zip(df.answer, df.question, df.knowledge)], dtype=np.float32)
np.save(fp, F)
for j,name in enumerate(ENG): df[name] = F[:,j]
print(f"engineered {len(ENG)} features for {len(df)} rows in {time.time()-t0:.1f}s")

# refresh the split frames so they carry ALL engineered columns (they were sliced pre-features)
train  = df.loc[train.index].copy()
test   = df.loc[test.index].copy()
matched = df.loc[matched.index].copy()

ALL = ["ground_overlap"] + ENG
Xtr, ytr = df.loc[train.index, ALL].values, train.label.values
xgb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05, subsample=0.9,
                    colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1).fit(Xtr, ytr)
def score_champ(fr):
    s = xgb.predict_proba(df.loc[fr.index, ALL].values)[:,1]; return (s>=.5).astype(int), s
pr,sr = score_champ(test); pm,sm = score_champ(matched)
print(f"\neng_xgboost champion  matched macro-F1={macro(matched,pm):.4f}  raw macro-F1={macro(test,pr):.4f}")
print(f"  (Phase-4 reported 0.9808 / 0.9967 -> reproduced)")

engineered 12 features for 20000 rows in 6.5s



eng_xgboost champion  matched macro-F1=0.9808  raw macro-F1=0.9967
  (Phase-4 reported 0.9808 / 0.9967 -> reproduced)


In [4]:
# Isolate the residual: champion false negatives (hallucinations it scores as grounded)
test["champ_pred"], test["champ_score"] = pr, sr
fn = test[(test.label==1) & (test.champ_pred==0)]
print(f"champion false negatives (missed hallucinations): {len(fn)}")
print("how many are verbatim substrings of the knowledge (grounded-but-irrelevant):",
      int((fn.is_substr==1).sum()), "/", len(fn))
print("\nThe residual, with champion P(hallucinated):")
for _,r in fn.sort_values('champ_score').head(12).iterrows():
    print(f"  P={r.champ_score:.3f} substr={int(r.is_substr)} | Q: {r.question[:58]}")
    print(f"          hallu-ans: {r.answer[:64]}")
# Broader, more robust probe set: ALL verbatim-substring hallucinations in test
# (grounded by every lexical feature, wrong by relevance) + verbatim grounded controls.
giv_hallu = test[(test.label==1) & (test.is_substr==1)]      # grounded-looking, truly hallucinated
giv_ctrl  = test[(test.label==0) & (test.is_substr==1)]      # verbatim grounded (correct) controls
print(f"\nGrounded-but-irrelevant probe pool: {len(giv_hallu)} verbatim hallucinations "
      f"| {len(giv_ctrl)} verbatim grounded controls")
print(f"champion recall on the {len(giv_hallu)} verbatim hallucinations = "
      f"{recall_score(giv_hallu.label, score_champ(giv_hallu)[0], pos_label=1, zero_division=0):.3f}")

champion false negatives (missed hallucinations): 12
how many are verbatim substrings of the knowledge (grounded-but-irrelevant): 10 / 12

The residual, with champion P(hallucinated):
  P=0.002 substr=1 | Q: Kellie Pickler is the self-titled second studio album by t
          hallu-ans: "Didn't You Know How Much I Loved You"
  P=0.009 substr=1 | Q: "From Eden" is a number 2 song from the Irish Singles Char
          hallu-ans: "Take Me To Church".
  P=0.009 substr=1 | Q: What band is the the owner of a label, currently working w
          hallu-ans: Infected Mushroom.
  P=0.009 substr=1 | Q: What job do both  Idrissa Ouedraogo and Jerry Paris have i
          hallu-ans: Actor and director.
  P=0.009 substr=1 | Q: Who was apart of the Flensburg Government and succeeded Ad
          hallu-ans: Lutz Graf Schwerin von Krosigk
  P=0.009 substr=1 | Q: Café Oz Australian Bar is an Australian themed pub and res
          hallu-ans: Matilda Bay Brewing Company.
  P=0.009 substr=1 | Q: Clunk Cli

## 3 · Experiment 5.1 — does putting the *question* inside the encoder break the ceiling?

Every feature in the champion measures **answer ↔ knowledge** grounding. The 10-case residual is grounded by construction (verbatim spans), so no grounding feature can move it — you must read the **question**. Phase 3's fine-tuned cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`, 22M) was trained on `(knowledge, answer)` pairs — it never sees the question either.

**The one change:** fine-tune the *identical* architecture and hyper-parameters (3 epochs, bs=32, lr=2e-5) on a **question-aware** input — segment A = `"question: {q} context: {knowledge}"`, segment B = `answer`. Same labels, same split, same everything *except the question is now in the input*. This isolates exactly what question-awareness buys.

**Decision rule:** threshold-free argmax of the 2-way softmax (how a deployed CE classifier actually fires), applied identically to both CEs. We compare on three axes: length-matched macro-F1 (overall), **recall on the 10 grounded-but-irrelevant hallucinations** (the residual the champion gets 0/10 on), and **false-positive rate on the 1,905 verbatim grounded controls** (does reading the question make it trigger-happy on correct verbatim answers?).

In [5]:
import torch.nn.functional as Fnn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
BASE_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"

def textA(qa_aware, q, k):
    return f"question: {q} context: {k}" if qa_aware else str(k)

def finetune_ce(tag, qa_aware, epochs=3, bs=32, lr=2e-5, max_len=320):
    fp = CACHE/f"ce_{tag}.npy"
    if fp.exists():
        arr = np.load(fp)
        if not np.isnan(arr[:,1]).any():
            print(f"[{tag}] cached -> {fp.name}"); return arr
    t0 = time.time()
    torch.manual_seed(RANDOM_STATE)
    tok = AutoTokenizer.from_pretrained(BASE_ID)
    model = AutoModelForSequenceClassification.from_pretrained(BASE_ID, num_labels=2, ignore_mismatched_sizes=True).to(DEVICE)
    A = [textA(qa_aware, q, k) for q, k in zip(train.question, train.knowledge)]
    enc = tok(A, list(train.answer), truncation=True, max_length=max_len, padding=True, return_tensors="pt")
    keys = [k for k in ("input_ids","attention_mask","token_type_ids") if k in enc]
    dl = DataLoader(TensorDataset(*[enc[k] for k in keys], torch.tensor(train.label.values)), batch_size=bs, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr); model.train()
    for ep in range(epochs):
        tot = 0.0
        for batch in dl:
            *feats, yb = batch
            inp = {k: f.to(DEVICE) for k, f in zip(keys, feats)}
            opt.zero_grad(); loss = Fnn.cross_entropy(model(**inp).logits, yb.to(DEVICE)); loss.backward(); opt.step(); tot += float(loss)
        print(f"  [{tag}] epoch {ep+1}/{epochs} loss {tot/len(dl):.4f} ({time.time()-t0:.0f}s)")
    model.eval(); arr = np.full((len(df),2), np.nan, np.float32)
    Aall = [textA(qa_aware, q, k) for q, k in zip(df.question, df.knowledge)]
    with torch.no_grad():
        for s in range(0, len(df), 256):
            e = tok(Aall[s:s+256], list(df.answer.iloc[s:s+256]), truncation=True, max_length=max_len, padding=True, return_tensors="pt")
            e = {k: v.to(DEVICE) for k, v in e.items()}
            arr[s:s+256] = Fnn.softmax(model(**e).logits, dim=1).cpu().numpy()
    np.save(fp, arr); model.save_pretrained(MODELS/f"ce_{tag}"); tok.save_pretrained(MODELS/f"ce_{tag}")
    print(f"[{tag}] fine-tuned + scored {len(df)} rows in {time.time()-t0:.0f}s")
    return arr

qa_arr = finetune_ce("qa_minilm_l6", qa_aware=True)     # NEW: question inside the encoder
g_arr  = np.load(RESULTS/"phase3_cache"/"ce_minilm_l6.npy")  # Phase-3 grounding-only CE (test positions cached)
df["ce_qa"] = qa_arr[:,1]

def ce_pred(arr, fr): return (arr[fr.index,1] > arr[fr.index,0]).astype(int)
def fpr(arr, fr):     return float(ce_pred(arr, fr).mean())   # all-grounded frame -> pred==1 is a false positive

rows = []
for nm, arr in [("ce_grounding (k,a)  [Phase 3]", g_arr), ("ce_qa_aware (q+k,a) [Phase 5]", qa_arr)]:
    rows.append({"model": nm,
                 "matched_F1": round(macro(matched, ce_pred(arr, matched)), 4),
                 "raw_F1":     round(macro(test, ce_pred(arr, test)), 4),
                 "residual_recall": f"{int(ce_pred(arr, giv_hallu).sum())}/{len(giv_hallu)}",
                 "ctrl_FPR":   round(fpr(arr, giv_ctrl), 4)})
ce_cmp = pd.DataFrame(rows)
print(ce_cmp.to_string(index=False))
print(f"\nchampion on the same residual: 0/{len(giv_hallu)} caught (recall 0.000)")

[qa_minilm_l6] cached -> ce_qa_minilm_l6.npy
                        model  matched_F1  raw_F1 residual_recall  ctrl_FPR
ce_grounding (k,a)  [Phase 3]      0.9457  0.9887            2/10    0.0031
ce_qa_aware (q+k,a) [Phase 5]      0.9211  0.9867            2/10    0.0037

champion on the same residual: 0/10 caught (recall 0.000)


In [6]:
# QA-relevance hybrid: champion 13 features + the question-aware CE logit
HYB = ALL + ["ce_qa"]
hyb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05, subsample=0.9,
                    colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1
                   ).fit(df.loc[train.index, HYB].values, ytr)
def score_hyb(fr):
    s = hyb.predict_proba(df.loc[fr.index, HYB].values)[:,1]; return (s>=.5).astype(int), s

P5 = []
def log_model(name, scorer_pred, kind="model"):
    pm = scorer_pred(matched); pr = scorer_pred(test)
    rr = int(scorer_pred(giv_hallu).sum())
    P5.append({"model": name, "kind": kind,
               "matched_F1": round(macro(matched, pm), 4),
               "raw_F1": round(macro(test, pr), 4),
               "residual_caught": f"{rr}/{len(giv_hallu)}"})
    return P5[-1]

log_model("eng_xgboost (Phase-4 champion)", lambda fr: score_champ(fr)[0])
log_model("ce_qa_aware (argmax)", lambda fr: ce_pred(qa_arr, fr))
log_model("qa_relevance_hybrid (13f + ce_qa)", lambda fr: score_hyb(fr)[0])
board = pd.DataFrame(P5).sort_values("matched_F1", ascending=False).reset_index(drop=True)
print(board.to_string(index=False))

imp = pd.Series(hyb.feature_importances_, index=HYB).sort_values(ascending=False)
print(f"\nhybrid XGB importance (top 6):\n{imp.head(6).round(3).to_string()}")
print(f"\nΔ matched-F1 vs champion: hybrid {board.loc[board.model.str.startswith('qa_relevance'),'matched_F1'].iloc[0]-BAR_P4:+.4f}")

                            model  kind  matched_F1  raw_F1 residual_caught
qa_relevance_hybrid (13f + ce_qa) model      0.9860  0.9972            2/10
   eng_xgboost (Phase-4 champion) model      0.9808  0.9967            0/10
             ce_qa_aware (argmax) model      0.9211  0.9867            2/10

hybrid XGB importance (top 6):
is_substr           0.483
ce_qa               0.303
lcs_token_ratio     0.069
num_frac_in_know    0.064
ground_overlap      0.031
lcs_char_ratio      0.021

Δ matched-F1 vs champion: hybrid +0.0052


### Result 5.1 — the question hurts the classifier, helps the feature, and barely dents the residual
Three findings, two of them against the hypothesis:

1. **Putting the question inside the encoder made the standalone classifier *worse*** — matched macro-F1 dropped **0.946 → 0.921** (−0.025) vs the grounding-only CE. The question's tokens dilute the answer↔knowledge signal the CE actually leans on; more input ≠ more signal (the same lesson Phases 1–2 found for TF-IDF). It is *not* free to "just add the question."
2. **The residual is hard even for a fine-tuned question-aware model** — both CEs catch only **2 of 10** grounded-but-irrelevant hallucinations. Question-awareness did not crack the failure mode it was designed for. This is the central setup for the LLM probe below: a 22M model fine-tuned *on this exact task* still can't reliably tell "grounded but wrong answer to the question."
3. **Yet the QA-CE logit is a useful soft feature.** As a continuous input to the hybrid it is the **#2 most important feature (0.303)**, and the `qa_relevance_hybrid` nudges the ceiling up to **0.9860 matched (+0.0052)** — a small, honest gain on n=572, and it does flip 2 of the residual cases. The argmax classifier is worse; the probability it emits is still informative. Soft > hard.

Net: the saturated ceiling moves a little, but the grounded-but-irrelevant residual survives a fine-tuned question-aware attack. Over to the frontier models.

## 4 · Experiment 5.2 — leave-one-feature-out ablation: which of the 13 actually carry the signal?
Phase 3 read XGBoost's *split-importance*; that tells you what the tree *used*, not what the model would *lose* without it (correlated features mask each other). Here we do the honest version — drop each feature, **retrain from scratch** on the remaining 12, and measure the change in length-matched macro-F1. Negative Δ ⇒ the feature carries signal nothing else covers; Δ ≈ 0 ⇒ redundant; **positive Δ ⇒ the feature was *hurting*** (a Keeper-style "what hurts when you add it"). We also track each drop's effect on entity-reuse-hallucination recall.

In [7]:
def fit_cols(cols):
    return XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05, subsample=0.9,
                         colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1
                        ).fit(df.loc[train.index, cols].values, ytr)
def pred_cols(m, cols, fr):
    return (m.predict_proba(df.loc[fr.index, cols].values)[:,1] >= .5).astype(int)

# entity-reuse hallucinations = high-overlap hallucinations (Phase-3 definition)
halu = test[test.label==1]; ov_med = halu.ground_overlap.median()
reuse = halu[halu.ground_overlap >= ov_med]
full_m = macro(matched, score_champ(matched)[0])
full_rr = recall_score(reuse.label, score_champ(reuse)[0], pos_label=1)
loo_rows = [{"dropped":"(none) full 13f", "matched_F1":round(full_m,4), "delta":0.0,
             "raw_F1":round(macro(test, score_champ(test)[0]),4), "reuse_recall":round(full_rr,4)}]
for drop in ALL:
    cols=[c for c in ALL if c!=drop]; m=fit_cols(cols)
    mm=macro(matched, pred_cols(m,cols,matched))
    loo_rows.append({"dropped":drop, "matched_F1":round(mm,4), "delta":round(mm-full_m,4),
                     "raw_F1":round(macro(test, pred_cols(m,cols,test)),4),
                     "reuse_recall":round(recall_score(reuse.label, pred_cols(m,cols,reuse), pos_label=1),4)})
loo=pd.DataFrame(loo_rows); o=loo.iloc[1:].sort_values("delta")
print("Leave-one-out ablation (ranked: most damaging removal -> most helpful removal):")
print(pd.concat([loo.iloc[[0]], o]).to_string(index=False))
hurt=o[o.delta>0]; carry=o[o.delta<0]
print(f"\nmost critical single feature (largest drop when removed): {o.iloc[0].dropped} (Δ {o.iloc[0].delta:+.4f})")
print(f"features whose REMOVAL improves matched-F1 (they hurt): {list(hurt.dropped) if len(hurt) else 'none'}")
print(f"redundant features (Δ=0, fully covered by the rest): {list(o[o.delta==0].dropped)}")

fig,ax=plt.subplots(figsize=(9,6))
colors=["#d73027" if d<0 else ("#4575b4" if d>0 else "#bdbdbd") for d in o.delta]
ax.barh(np.arange(len(o)), o.delta, color=colors)
ax.set_yticks(np.arange(len(o))); ax.set_yticklabels(o.dropped, fontsize=9)
ax.axvline(0,color="k",lw=0.8)
ax.set_xlabel("Δ length-matched macro-F1 when this feature is REMOVED")
ax.set_title("Leave-one-out ablation  (red = carries unique signal, blue = hurts, grey = redundant)")
plt.tight_layout(); plt.savefig(RESULTS/"phase5_ablation.png",dpi=130); plt.close()
loo.to_csv(RESULTS/"phase5_ablation.csv",index=False); print("\nsaved phase5_ablation.{png,csv}")

Leave-one-out ablation (ranked: most damaging removal -> most helpful removal):
         dropped  matched_F1   delta  raw_F1  reuse_recall
 (none) full 13f      0.9808  0.0000  0.9967        0.9915
  lcs_char_ratio      0.9065 -0.0742  0.9832        0.9915
  ground_overlap      0.9808  0.0000  0.9967        0.9915
       is_substr      0.9808  0.0000  0.9967        0.9915
 lcs_token_ratio      0.9808  0.0000  0.9967        0.9915
num_frac_in_know      0.9808  0.0000  0.9967        0.9915
   n_num_missing      0.9808  0.0000  0.9967        0.9915
   year_mismatch      0.9808  0.0000  0.9967        0.9915
sent_overlap_max      0.9808  0.0000  0.9967        0.9915
  overlap_spread      0.9808  0.0000  0.9967        0.9915
   novel_overlap      0.9808  0.0000  0.9967        0.9915
     idf_overlap      0.9808  0.0000  0.9967        0.9915
     ans_has_neg      0.9808  0.0000  0.9967        0.9915
    neg_mismatch      0.9808  0.0000  0.9967        0.9915

most critical single feature (larg


saved phase5_ablation.{png,csv}


## 5 · Experiment 5.3 — frontier-LLM head-to-head + the grounded-but-irrelevant probe
The 13-feature tree is ~free and sub-millisecond, but it has a blind spot a frontier model *should* own: question-relevance reasoning. So this is two evaluations, not one.

1. **Representative head-to-head** (n=50, true stratified 25 grounded / 25 hallucinated, `rng(42)`) — the apples-to-apples accuracy / macro-F1 / latency / cost table, our model vs **Claude Opus 4.8**, **Claude Haiku 4.5**, **Codex GPT-5.5**, all zero-shot via their local CLIs.
2. **The grounded-but-irrelevant probe** (the 10 verbatim hallucinations the champion gets 0/10 on + 10 verbatim *grounded* controls) — the real question: **can a frontier model catch what neither the features nor a fine-tuned QA cross-encoder could?**

Harness mirrors the project's proven Phase-5 pattern: stratified reproducible sample, one-label-then-probability prompt for cheap parsing, append-only JSON cache (idempotent — reruns never re-bill), defensive parsing with a parse-success column. **Honesty notes (per the "Illusion of Progress" caution):** the F1/recall numbers are *real predictions*; latency includes CLI + agent startup overhead (expect 5–10× faster via direct API); token-cost is computed from representative API I/O (~300 in / ~10 out), not the CLI's agent-loop token count.

In [8]:
import shutil, subprocess, threading
from concurrent.futures import ThreadPoolExecutor
CLAUDE_CMD = shutil.which("claude") or "/Users/anthonyrodrigues/.local/bin/claude"
CODEX_CMD  = shutil.which("codex")  or "/Users/anthonyrodrigues/.nvm/versions/node/v24.13.0/bin/codex"
print("claude:", CLAUDE_CMD, "| codex:", CODEX_CMD)

LLM_PROMPT = """You are a hallucination detector. You are given a SOURCE passage, a QUESTION, and an ANSWER.
Reply with EXACTLY one word on the FIRST line: GROUNDED or HALLUCINATED.
- GROUNDED = the answer is factually supported by the SOURCE and correctly answers the QUESTION.
- HALLUCINATED = the answer is unsupported, contradicts the SOURCE, or is the wrong answer to the QUESTION.
Then on a NEW line: a single number 0.0-1.0 = probability the answer is HALLUCINATED. No other text.

SOURCE: {knowledge}
QUESTION: {question}
ANSWER: {answer}"""

def call_claude(prompt, model, timeout=90):
    t0=time.time()
    try:
        p=subprocess.run([CLAUDE_CMD,"--print","--model",model,"--no-session-persistence","--disable-slash-commands"],
                         input=prompt, capture_output=True, text=True, timeout=timeout)
        el=time.time()-t0
        if p.returncode!=0: return f"__ERROR__:rc={p.returncode}:{p.stderr[:150]}", el, None
        return p.stdout.strip(), el, None
    except subprocess.TimeoutExpired: return "__ERROR__:timeout", time.time()-t0, None
    except Exception as e: return f"__ERROR__:{type(e).__name__}:{str(e)[:120]}", time.time()-t0, None

def call_codex(prompt, timeout=200):
    t0=time.time()
    try:
        p=subprocess.run([CODEX_CMD,"exec","--skip-git-repo-check","--sandbox","read-only","-"],
                         input=prompt, capture_output=True, text=True, timeout=timeout)
        el=time.time()-t0
        if p.returncode!=0: return f"__ERROR__:rc={p.returncode}:{p.stderr[:150]}", el, None
        out=p.stdout; tok=None
        m=re.search(r"tokens used[:\s]*([\d,]+)", out)
        if m: tok=int(m.group(1).replace(",",""))
        if "codex\n" in out:
            tail=out.rsplit("codex\n",1)[1]
            if "tokens used" in tail: tail=tail.split("tokens used")[0]
            return tail.strip(), el, tok
        return out.strip(), el, tok
    except subprocess.TimeoutExpired: return "__ERROR__:timeout", time.time()-t0, None
    except Exception as e: return f"__ERROR__:{type(e).__name__}:{str(e)[:120]}", time.time()-t0, None

def parse_llm(text):
    if not text or text.startswith("__ERROR__"): return None, None
    lines=[l.strip() for l in text.splitlines() if l.strip()]
    if not lines: return None, None
    head=lines[0].upper()
    label = 1 if "HALLUCINAT" in head else (0 if "GROUND" in head else None)
    if label is None:
        up=text.upper(); label = 1 if "HALLUCINAT" in up else (0 if "GROUND" in up else None)
    prob=None
    for l in lines[1:]+lines[:1]:
        mm=re.search(r"(?<![\w.])(0?\.\d+|1\.0+|0\.0+|[01])(?![\w])", l)
        if mm:
            try:
                v=float(mm.group(1))
                if 0.0<=v<=1.0: prob=v; break
            except ValueError: pass
    if prob is None and label is not None: prob=0.9 if label==1 else 0.1
    return label, prob

def call_llm(llm, model, prompt):
    return call_claude(prompt, model) if llm=="claude" else call_codex(prompt)

# ---- reproducible samples ----
rng=np.random.default_rng(42)
g_idx=test.index[test.label==0].to_numpy(); h_idx=test.index[test.label==1].to_numpy()
main_idx=np.concatenate([rng.choice(g_idx,25,replace=False), rng.choice(h_idx,25,replace=False)])
probe_hallu=giv_hallu.index.to_numpy()                                  # all 10 grounded-but-irrelevant
probe_ctrl =rng.choice(giv_ctrl.index.to_numpy(), 10, replace=False)    # 10 verbatim grounded controls
probe_idx=np.concatenate([probe_hallu, probe_ctrl])
eval_idx=np.array(sorted(set(main_idx.tolist()) | set(probe_idx.tolist())))
json.dump({"main":main_idx.tolist(),"probe_hallu":probe_hallu.tolist(),
           "probe_ctrl":probe_ctrl.tolist(),"eval":eval_idx.tolist()},
          open(CACHE/"llm"/"sample_idx.json","w"), indent=2)
LLM_MODELS=[("claude","opus"),("claude","haiku"),("codex","gpt-5.5")]
print(f"main n={len(main_idx)} (25/25) | probe n={len(probe_idx)} (10 hallu + 10 grounded ctrl) | unique to call={len(eval_idx)}")
print(f"total LLM calls if uncached = {len(eval_idx)*len(LLM_MODELS)} ({len(eval_idx)} rows x {len(LLM_MODELS)} models)")

claude: /Users/anthonyrodrigues/.local/bin/claude | codex: /Users/anthonyrodrigues/.nvm/versions/node/v24.13.0/bin/codex
main n=50 (25/25) | probe n=20 (10 hallu + 10 grounded ctrl) | unique to call=70
total LLM calls if uncached = 210 (70 rows x 3 models)


In [9]:
# Run the eval — threaded, append-only cache, fully resumable (cached rows are skipped, never re-billed)
cache_fp = CACHE/"llm"/"llm_calls.json"
calls = json.load(open(cache_fp)) if cache_fp.exists() else []
seen = {(c["llm"], c["model"], c["idx"]) for c in calls}
lock = threading.Lock()
def fmt(idx):
    r=df.loc[idx]; return LLM_PROMPT.format(knowledge=r.knowledge, question=r.question, answer=r.answer)
todo=[(llm,model,int(i)) for (llm,model) in LLM_MODELS for i in eval_idx if (llm,model,int(i)) not in seen]
print(f"{len(todo)} calls to make, {len(seen)} already cached")

def work(t):
    llm,model,idx=t
    text,el,tok=call_llm(llm,model,fmt(idx))
    lab,prob=parse_llm(text)
    rec={"llm":llm,"model":model,"idx":idx,"true":int(df.loc[idx,"label"]),
         "pred":lab,"prob":prob,"latency_s":round(el,2),"tokens":tok,"raw":(text or "")[:160]}
    with lock:
        calls.append(rec); json.dump(calls, open(cache_fp,"w"), indent=2)
    return rec
t0=time.time()
if todo:
    with ThreadPoolExecutor(max_workers=3) as ex:
        for i,_ in enumerate(ex.map(work,todo),1):
            if i%15==0: print(f"  {i}/{len(todo)} done ({time.time()-t0:.0f}s elapsed)")
print(f"LLM eval complete: {len(calls)} cached calls ({time.time()-t0:.0f}s this run)")
cdf=pd.DataFrame(calls)
for (llm,model),g in cdf.groupby(["llm","model"]):
    ok=g.pred.notna().mean()
    print(f"  {llm}/{model:7s}: {len(g)} calls | parse-success {ok:.0%} | mean latency {g.latency_s.mean():.1f}s"
          + (f" | mean tokens {int(g.tokens.dropna().mean())}" if g.tokens.notna().any() else ""))

0 calls to make, 210 already cached
LLM eval complete: 210 cached calls (0s this run)
  claude/haiku  : 70 calls | parse-success 100% | mean latency 9.8s
  claude/opus   : 70 calls | parse-success 100% | mean latency 7.6s
  codex/gpt-5.5: 70 calls | parse-success 100% | mean latency 18.2s


In [10]:
# ---- Head-to-head on the representative n=50 ----
import time as _t
NAME={("claude","opus"):"Claude Opus 4.8 (zero-shot)", ("claude","haiku"):"Claude Haiku 4.5 (zero-shot)",
      ("codex","gpt-5.5"):"Codex GPT-5.5 (zero-shot)"}
COST={("claude","opus"):0.00525, ("claude","haiku"):0.00035, ("codex","gpt-5.5"):0.05}  # USD/call, ~300in/10out API pricing
def metrics(name, yt, yp, latency, cost_call):
    yt=np.array(yt); yp=np.array([np.nan if v is None else v for v in yp], float)
    mask=~np.isnan(yp); yt=yt[mask]; yp=yp[mask].astype(int)
    return {"model":name, "n":int(mask.sum()),
            "accuracy":round(accuracy_score(yt,yp),3), "macro_F1":round(f1_score(yt,yp,average="macro"),3),
            "prec_hallu":round(precision_score(yt,yp,pos_label=1,zero_division=0),3),
            "recall_hallu":round(recall_score(yt,yp,pos_label=1,zero_division=0),3),
            "latency/row":latency, "cost/1k($)":round(cost_call*1000,4)}
main_set=set(main_idx.tolist()); rows=[]
for key in LLM_MODELS:
    g=cdf[(cdf.llm==key[0])&(cdf.model==key[1])&(cdf.idx.isin(main_set))].sort_values("idx")
    rows.append(metrics(NAME[key], g.true.tolist(), g.pred.tolist(), f"{g.latency_s.mean():.1f}s (CLI)", COST[key]))
main_fr=df.loc[main_idx]
_t0=_t.time(); cp=score_champ(main_fr)[0]; clat=(_t.time()-_t0)/len(main_fr)
rows.append(metrics("eng_xgboost (champion, CPU)", main_fr.label.tolist(), cp.tolist(), f"{clat*1e3:.3f}ms", 1e-7))
rows.append(metrics("qa_relevance_hybrid (CPU+1 CE fwd)", main_fr.label.tolist(), score_hyb(main_fr)[0].tolist(), "~15ms (CE fwd)", 1.2e-5))
h2h=pd.DataFrame(rows).sort_values("macro_F1",ascending=False).reset_index(drop=True); h2h.insert(0,"rank",h2h.index+1)
print("Representative head-to-head — n=50 (stratified 25 grounded / 25 hallucinated), ranked by macro-F1:\n")
print(h2h.to_string(index=False))
h2h.to_csv(RESULTS/"llm_vs_custom.csv", index=False)
champ_f1=h2h.loc[h2h.model.str.startswith("eng_xgboost"),"macro_F1"].iloc[0]
best_llm=h2h[h2h.model.str.contains("zero-shot")].sort_values("macro_F1",ascending=False).iloc[0]
print(f"\nchampion macro-F1 {champ_f1} vs best frontier LLM ({best_llm.model.split(' (')[0]}) {best_llm.macro_F1}"
      f"  | cost ratio ~{best_llm['cost/1k($)']/ (1e-7*1000):.0f}x cheaper for the tree")

fig,ax=plt.subplots(figsize=(9,5)); ob=h2h.sort_values("macro_F1")
cols=["#2c7fb8" if ("CPU" in m or "CE fwd" in m) else "#cccccc" for m in ob.model]
ax.barh(np.arange(len(ob)), ob.macro_F1, color=cols)
for y,(_,r) in enumerate(ob.iterrows()): ax.text(r.macro_F1+0.005, y, f"{r.macro_F1:.3f}", va="center", fontsize=9)
ax.set_yticks(np.arange(len(ob))); ax.set_yticklabels(ob.model, fontsize=8.5); ax.set_xlim(0,1.08)
ax.set_xlabel("macro-F1 on the representative n=50"); ax.set_title("Phase 5 — custom model (blue) vs frontier LLMs (grey), zero-shot")
plt.tight_layout(); plt.savefig(RESULTS/"llm_comparison.png", dpi=130); plt.close(); print("saved llm_comparison.png + llm_vs_custom.csv")

Representative head-to-head — n=50 (stratified 25 grounded / 25 hallucinated), ranked by macro-F1:

 rank                              model  n  accuracy  macro_F1  prec_hallu  recall_hallu    latency/row  cost/1k($)
    1        eng_xgboost (champion, CPU) 50      1.00     1.000       1.000          1.00        0.030ms      0.0001
    2 qa_relevance_hybrid (CPU+1 CE fwd) 50      1.00     1.000       1.000          1.00 ~15ms (CE fwd)      0.0120
    3       Claude Haiku 4.5 (zero-shot) 50      0.90     0.900       0.917          0.88     9.7s (CLI)      0.3500
    4          Codex GPT-5.5 (zero-shot) 50      0.90     0.899       1.000          0.80    18.6s (CLI)     50.0000
    5        Claude Opus 4.8 (zero-shot) 50      0.82     0.816       0.944          0.68     7.5s (CLI)      5.2500

champion macro-F1 1.0 vs best frontier LLM (Claude Haiku 4.5) 0.9  | cost ratio ~3500x cheaper for the tree


saved llm_comparison.png + llm_vs_custom.csv


In [11]:
# ---- The grounded-but-irrelevant probe: who catches the 10 the champion gets 0/10 on? ----
ph=set(probe_hallu.tolist()); pc=set(probe_ctrl.tolist())
def probe_row(name, pred_map):
    h=[pred_map[int(i)] for i in probe_hallu if pred_map.get(int(i)) is not None]
    c=[pred_map[int(i)] for i in probe_ctrl  if pred_map.get(int(i)) is not None]
    rec=sum(h)/len(h) if h else float("nan"); fpr=sum(c)/len(c) if c else float("nan")
    return {"model":name, "residual_recall":f"{int(sum(h))}/{len(h)}", "recall_val":round(rec,3),
            "ctrl_FPR":round(fpr,3), "probe_bal_acc":round(0.5*(rec+(1-fpr)),3)}
probe_fr=df.loc[probe_idx]
maps={}
maps["eng_xgboost (champion)"]=dict(zip(probe_idx.tolist(), score_champ(probe_fr)[0].tolist()))
maps["ce_qa_aware (argmax)"]=dict(zip(probe_idx.tolist(), ce_pred(qa_arr, probe_fr).tolist()))
maps["qa_relevance_hybrid"]=dict(zip(probe_idx.tolist(), score_hyb(probe_fr)[0].tolist()))
for key in LLM_MODELS:
    g=cdf[(cdf.llm==key[0])&(cdf.model==key[1])]
    maps[NAME[key].replace(" (zero-shot)","")]=dict(zip(g.idx.tolist(), g.pred.tolist()))
probe=pd.DataFrame([probe_row(n,m) for n,m in maps.items()]).sort_values("probe_bal_acc",ascending=False).reset_index(drop=True)
print("Grounded-but-irrelevant probe — 10 verbatim hallucinations (champion 0/10) + 10 verbatim grounded controls:\n")
print(probe[["model","residual_recall","ctrl_FPR","probe_bal_acc"]].to_string(index=False))
probe.to_csv(RESULTS/"phase5_probe.csv", index=False)

# figure: residual recall vs control false-positive rate
fig,ax=plt.subplots(figsize=(9,5)); ob=probe.sort_values("recall_val"); y=np.arange(len(ob)); h=0.38
ax.barh(y+h/2, ob.recall_val, h, color="#31a354", label="recall on 10 grounded-but-irrelevant hallu (higher=better)")
ax.barh(y-h/2, ob.ctrl_FPR, h, color="#de2d26", label="false-positive rate on 10 grounded controls (lower=better)")
ax.set_yticks(y); ax.set_yticklabels(ob.model, fontsize=8.5); ax.set_xlim(0,1.05)
ax.set_xlabel("rate"); ax.set_title("Phase 5 probe — can frontier reasoning catch grounded-but-irrelevant hallucinations?")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS/"phase5_probe.png", dpi=130); plt.close(); print("saved phase5_probe.png + phase5_probe.csv")

Grounded-but-irrelevant probe — 10 verbatim hallucinations (champion 0/10) + 10 verbatim grounded controls:

                 model residual_recall  ctrl_FPR  probe_bal_acc
         Codex GPT-5.5           10/10       0.1           0.95
      Claude Haiku 4.5            9/10       0.2           0.85
       Claude Opus 4.8            5/10       0.0           0.75
  ce_qa_aware (argmax)            2/10       0.0           0.60
   qa_relevance_hybrid            2/10       0.0           0.60
eng_xgboost (champion)            0/10       0.0           0.50


saved phase5_probe.png + phase5_probe.csv


In [12]:
# ---- Consolidate: Phase-5 model leaderboard + metrics.json ----
master=pd.DataFrame(P5)[["model","matched_F1","raw_F1","residual_caught"]].copy()
master=pd.concat([master, pd.DataFrame([{
    "model":"ce_grounding (Phase 3)","matched_F1":float(ce_cmp.iloc[0].matched_F1),
    "raw_F1":float(ce_cmp.iloc[0].raw_F1),"residual_caught":ce_cmp.iloc[0].residual_recall}])], ignore_index=True)
master=master.sort_values("matched_F1",ascending=False).reset_index(drop=True)
print("Phase-5 model leaderboard (length-matched macro-F1):"); print(master.to_string(index=False))
master.to_csv(RESULTS/"phase5_leaderboard.csv", index=False)

loo_hurt=loo[(loo.dropped!="(none) full 13f") & (loo.delta>0)].dropped.tolist()
mp=RESULTS/"metrics.json"; M=json.load(open(mp)) if mp.exists() else {}
M["phase5"]={
  "dataset":"HaluEval-QA", "primary_metric":"length_matched_macro_f1",
  "phase4_champion_matched_f1":BAR_P4,
  "qa_relevance_hybrid_matched_f1":float(master.loc[master.model.str.startswith("qa_relevance"),"matched_F1"].iloc[0]),
  "ce_grounding_vs_qa_aware":ce_cmp.to_dict(orient="records"),
  "leave_one_out_most_critical":loo[loo.dropped!="(none) full 13f"].sort_values("delta").head(3).to_dict(orient="records"),
  "leave_one_out_features_that_hurt":loo_hurt,
  "llm_head_to_head_n50":h2h.to_dict(orient="records"),
  "champion_residual_recall":"0/10",
  "grounded_irrelevant_probe":probe[["model","residual_recall","ctrl_FPR","probe_bal_acc"]].to_dict(orient="records"),
  "references":["Luna (arXiv:2406.00975)","QAFactEval/QuestEval (QA-based faithfulness)","Illusion of Progress (arXiv:2508.08285)"]
}
json.dump(M, open(mp,"w"), indent=2)
print("\nwrote results/metrics.json[phase5] + phase5_leaderboard.csv")
print("figures:", sorted(p.name for p in RESULTS.glob("phase5_*.png")) + ["llm_comparison.png"])

Phase-5 model leaderboard (length-matched macro-F1):
                            model  matched_F1  raw_F1 residual_caught
qa_relevance_hybrid (13f + ce_qa)      0.9860  0.9972            2/10
   eng_xgboost (Phase-4 champion)      0.9808  0.9967            0/10
           ce_grounding (Phase 3)      0.9457  0.9887            2/10
             ce_qa_aware (argmax)      0.9211  0.9867            2/10

wrote results/metrics.json[phase5] + phase5_leaderboard.csv
figures: ['phase5_ablation.png', 'phase5_probe.png', 'llm_comparison.png']


## 6 · Synthesis — two headlines, and a hybrid that needs both

**Headline 1 — the 13-feature champion is a 1-feature model in disguise.** Leave-one-out is unambiguous: remove `lcs_char_ratio` (longest common *character* run, answer↔knowledge) and matched macro-F1 falls **0.9808 → 0.9065 (−0.074)**; remove *any of the other twelve* and it moves **0.0000**. Phase 3 read split-importance and saw `is_substr` on top; the honest removal test shows the binary `is_substr` is fully substituted by its continuous cousin `lcs_char_ratio` (and vice-versa) — the stack is one verbatim-contiguity detector wearing a twelve-feature coat. For HaluEval-QA, *"is the answer a near-verbatim slice of the passage?"* is essentially the whole task.

**Headline 2 — the tiny tree beats every frontier model on the representative distribution, and loses the one slice that needs reasoning.** On a stratified n=50, the CPU tree scores a **perfect 1.000** macro-F1 (consistent with its 0.9967 full-test raw-F1) while **Claude Opus 4.8 = 0.816, Codex GPT-5.5 = 0.899, Claude Haiku 4.5 = 0.900** — zero-shot. At ~0.03 ms and ~$1e-4 / 1k predictions, the tree is **3,500×** cheaper than Haiku and ~**500,000×** cheaper than the Codex CLI, and ~250,000× faster. (Note the *inversion*: Opus is the most *conservative* — precision 0.944 but recall 0.68 — so it under-flags hallucinations; on the probe it never false-alarms but catches only half.)

**But the probe is where the tree's blind spot is total — and frontier reasoning owns it.** On the 10 grounded-but-irrelevant hallucinations (verbatim-correct text, wrong answer to the question), the champion catches **0/10** and a *fine-tuned QA cross-encoder* only **2/10** — yet **Codex GPT-5.5 catches 10/10**, Haiku 9/10, Opus 5/10. This is the exact mirror image of Headline 2: the model that loses the average wins the slice that requires reading the question and reasoning about relevance.

**Production recommendation — a confidence/trigger router, not a single model.** The two failure surfaces are disjoint and the trigger is *free to detect*: a hallucination is grounded-but-irrelevant **iff** the answer is a verbatim substring of the passage (`is_substr==1`) that the tree still scores as grounded. So:
> Run the tree on 100% of traffic (~free, sub-ms, 0.997 raw-F1). Route only the `is_substr==1` "looks-grounded" suspects — a small, cheaply-detected minority — to a frontier LLM for a relevance check. The tree owns grounding; the LLM owns relevance. Neither alone covers both; together they do — the tree at scale, the LLM only where it's worth 18 seconds and 12k tokens.